In [25]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
wine = load_wine()
#print(list(wine.keys()))
print(wine.feature_names)
df = pd.DataFrame(wine.data, columns=wine.feature_names)

#develop linear regression: Goal predict Alcohol content using the remaining 12 features , which 2 features influence alcohol the most

target="alcohol"

features = df.columns.drop(target)
X = df.drop(columns=["alcohol", "target"], errors="ignore")

#X = df[features]
y = df[target]
#print("X: ", list(X)) #print("Y:", list(y))
#Lin regre
from sklearn.linear_model import LinearRegression

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model = LinearRegression()
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)

#from sklearn.linear_model import LogisticRegression


['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


In [26]:
y = df["alcohol"]
X = df.drop(columns=["alcohol", "target"], errors="ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

#print(y_pred)

# Rank features by absolute standardized coefficient
importance = pd.DataFrame({
    "feature": features,
    "std_coef": model.coef_,
    "abs_influence": np.abs(model.coef_),
}).sort_values(by="abs_influence", ascending=False)

print("\nTop features by standardized influence:")
print(importance.to_string(index=False))

print("\nTop 2 features:")
print(importance.head(2))

Test R²:   0.6889
Test RMSE: 0.4309
Test MAE:  0.3675

Top features by standardized influence:
                     feature  std_coef  abs_influence
             color_intensity  0.394276       0.394276
                     proline  0.298278       0.298278
           alcalinity_of_ash -0.161299       0.161299
                  malic_acid  0.128679       0.128679
                         hue  0.094172       0.094172
od280/od315_of_diluted_wines  0.085861       0.085861
             proanthocyanins -0.083039       0.083039
                         ash  0.074655       0.074655
               total_phenols  0.058450       0.058450
                   magnesium -0.026295       0.026295
        nonflavanoid_phenols -0.015891       0.015891
                  flavanoids -0.010809       0.010809

Top 2 features:
            feature  std_coef  abs_influence
8   color_intensity  0.394276       0.394276
11          proline  0.298278       0.298278


In [34]:
#Classification Showdown
#Goal: Predict the wine cultivar ("Target": 0,1,2)

# import numpy as np
# import pandas as pd
# from sklearn.datasets import load_wine
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
# from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# wine = load_wine()

# print(wine.feature_names)
# df = pd.DataFrame(wine.data, columns=wine.feature_names)
# df["cultivar"] = wine.target
# y = df["cultivar"]
# X = df.drop(columns=["cultivar", "target"], errors="ignore")

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# from sklearn.linear_model import LogisticRegression

# model = LogisticRegression(max_iter=10000)
# model.fit(X_train, y_train)

# print(df.columns.tolist())

import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load features (13 chemical measurements) and target (cultivars 0, 1, 2)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

# 2. Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Build pipeline: scale features first, then fit multiclass LogisticRegression
clf_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# 4. Cross-validation on full dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf_pipe, X, y, cv=cv, scoring="accuracy")
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 5. Fit on training set and evaluate on held-out test set
clf_pipe.fit(X_train, y_train)
y_pred = clf_pipe.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(
    "\nClassification Report:\n",
    classification_report(y_test, y_pred, target_names=wine.target_names),
)

# 6. Inspect per-class standardized coefficients
model = clf_pipe.named_steps["logisticregression"]
coef_df = pd.DataFrame(model.coef_, columns=X.columns, index=wine.target_names).T
coef_df["max_abs_coef"] = coef_df.abs().max(axis=1)
print("\nTop 5 features by max |coef| across cultivars:")
print(coef_df.sort_values("max_abs_coef", ascending=False).head(5).round(4))

5-Fold CV Accuracy: 0.9833 (+/- 0.0136)
Test Accuracy: 0.9722
Test Macro F1: 0.9710

Confusion Matrix:
 [[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]

Classification Report:
               precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36


Top 5 features by max |coef| across cultivars:
                   class_0  class_1  class_2  max_abs_coef
color_intensity     0.2202  -1.0960   0.8758        1.0960
proline             0.9265  -1.0099   0.0834        1.0099
flavanoids          0.7053   0.1982  -0.9035        0.9035
alcohol             0.7175  -0.8615   0.1441        0.8615
alcalinity_of_ash  -0.8351   0.4662   0.3688        0.8351
